<div style="text-align:center; font-family:Tahoma, Arial; line-height:1.8;">

  <div style="font-size:42px; font-weight:bold; color:#0F8298;">
    machine learning 15
  </div>

  <div style="font-size:28px; font-weight:600; color:#6C3BAA; margin-top:8px;">
    Recommender System </div>
  

  <div style="font-size:18px; color:#4b4f9c;">
</div>
Collaborative Filtering

پالایش همکاری‌محور 

<hr style="width:60%; margin:20px auto; border:1px solid #ddd;">

<font color=497890 size=3>

اهداف: پس از اتمام این آزمایشگاه شما قادر خواهید بود

 یک سیستم توصیه‌گر مبتنی بر پالایش همکاری‌محور $( \text{collaborative filtering} )$ ایجاد کنید

<font color=GREEN size=5>
HAKAN Fatemi (www.hooko.ir)


____
</div> </div>

سیستم‌های توصیه‌گر مجموعه‌ای از الگوریتم‌ها هستند که برای توصیه اقلام به کاربران، بر اساس اطلاعات گرفته‌شده از کاربر، استفاده می‌شوند. این سیستم‌ها به‌شدت فراگیر شده‌اند و به‌طور معمول در فروشگاه‌های آنلاین، پایگاه‌های داده‌ی فیلم و کاریابی‌ها دیده می‌شوند. در این دفترچه، سیستم‌های توصیه‌گر مبتنی بر پالایش همکاری‌محور $( \text{Collaborative Filtering} )$ را بررسی کرده و یک نسخه‌ی ساده از آن را با استفاده از پایتون و کتابخانه‌ی پانداس پیاده‌سازی خواهیم کرد

<h1>فهرست مطالب</h1>

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="#ref1">(Acquiring the Data) دریافت داده</a></li>
        <li><a href="#ref2">(Preprocessing) پیش‌پردازش</a></li>
        <li><a href="#ref3">(Collaborative Filtering) پالایش همکاری‌محور</a></li>
    </ol>
</div>
<br>
<hr>

<a id="ref1"></a>

# $( \text{Acquiring the Data} )$ دریافت داده

In [128]:
# توجه: در صورت دانلود نشدن، به شکل دستی از آدرس داده شده دانلود کنید
import os
import urllib.request
import zipfile

url= "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%205/data/moviedataset.zip"

if not (os.path.exists("movies.csv") and os.path.exists("ratings.csv")):
    if not os.path.exists("moviedataset.zip"):
        urllib.request.urlretrieve(url, "moviedataset.zip")
    with zipfile.ZipFile("moviedataset.zip", "r") as zip_ref:
        zip_ref.extractall(".")
    print("دانلود و استخراج کامل شد")
else:
    print("فایل‌ها از قبل وجود دارند")

فایل‌ها از قبل وجود دارند


<hr>

<a id="ref2"></a>

# $( \text{Preprocessing} )$ پیش‌پردازش

ابتدا، بیایید تمام $( \text{import} )$های مورد نیاز را از سر راه برداریم (یعنی آن‌ها را در ابتدا وارد کنیم)

In [129]:
import pandas as pd
from math import sqrt
import numpy as np
import matplotlib.pyplot as plt

حال بیایید هر فایل را در دیتافریم مربوط به خود بخوانیم

In [130]:
# ذخیره‌ی اطلاعات فیلم‌ها در یک دیتافریم پانداس
movies_df = pd.read_csv("movies.csv")
# ذخیره‌ی اطلاعات کاربران در یک دیتافریم پانداس
ratings_df = pd.read_csv("ratings.csv")

بیایید نگاهی هم به نحوه‌ی سازمان‌دهی هر یک از آن‌ها بیندازیم

In [131]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


بیایید با استفاده از تابع $( \text{replace} )$ پانداس، سال را از ستون **$( \text{title} )$** حذف کرده و آن را در یک ستون جدید به نام **$( \text{year} )$** ذخیره کنیم.

In [132]:
# استفاده از عبارات منظم برای یافتن سال ذخیره‌شده بین پرانتز
# پرانتزها را مشخص می‌کنیم تا با فیلم‌هایی که در عنوان خود سال دارند تداخل پیدا نکنیم
movies_df["year"] = movies_df.title.str.extract("(\(\d\d\d\d\))", expand=False)
# حذف پرانتزها
movies_df["year"] = movies_df.year.str.extract("(\d\d\d\d)", expand=False)
# حذف سال‌ها از ستون "title"
movies_df["title"] = movies_df.title.str.replace("(\(\d\d\d\d\))", "", regex=True)
# اعمال تابع strip برای حذف هر گونه کاراکتر فضای خالی انتهایی که ممکن است ظاهر شده باشد
movies_df["title"] = movies_df["title"].apply(lambda x: x.strip())

بیایید نتیجه را ببینیم

In [133]:
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


با این کار، بیایید ستون $( \text{genres} )$ را نیز حذف کنیم، زیرا برای این سیستم توصیه‌گر خاص به آن نیازی نخواهیم داشت

In [134]:
# حذف ستون genres
movies_df = movies_df.drop("genres", axis=1)

در اینجا دیتافریم نهایی فیلم‌ها آورده شده است

In [135]:
movies_df.head()

,movieId,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995


در ادامه، بیایید نگاهی به دیتافریم $( \text{ratings} )$ بیندازیم

In [136]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,169,2.5,1204927694
1,1,2471,3.0,1204927438
2,1,48516,5.0,1204927435
3,2,2571,3.5,1436165433
4,2,109487,4.0,1436165496


هر سطر در دیتافریم $( \text{ratings} )$ دارای یک $( \text{user id} )$ مرتبط با حداقل یک فیلم، یک امتیاز $( \text{rating} )$ و یک $( \text{timestamp} )$ 

است که نشان می‌دهد کاربر چه زمانی آن را بررسی کرده است. ما به ستون $( \text{timestamp} )$ نیازی نخواهیم داشت

بنابراین بیایید آن را حذف کنیم تا حافظه آزاد شود

In [137]:
# یک سطر یا ستون مشخص را از دیتافریم حذف می‌کند Drop تابع
ratings_df = ratings_df.drop("timestamp", axis=1)

در اینجا دیتافریم نهایی $( \text{ratings} )$ به این شکل است

In [138]:
ratings_df.head()

,userId,movieId,rating
0,1,169,2.5
1,1,2471,3.0
2,1,48516,5.0
3,2,2571,3.5
4,2,109487,4.0


<hr>

<a id="ref3"></a>

# $( \text{Collaborative Filtering} )$ پالایش همکاری‌محور

اکنون زمان آن رسیده که کار خود را روی سیستم‌های توصیه‌گر شروع کنیم.

اولین تکنیکی که قرار است بررسی کنیم **پالایش همکاری‌محور $( \text{Collaborative Filtering} )$** نام دارد که با نام **پالایش کاربر-کاربر $( \text{User-User Filtering} )$** نیز شناخته می‌شود. همان‌طور که از نام جایگزین آن پیداست، این تکنیک از سایر کاربران برای توصیه اقلام به کاربر ورودی استفاده می‌کند. این تکنیک سعی می‌کند کاربرانی را پیدا کند که ترجیحات و نظرات مشابهی با کاربر ورودی دارند و سپس اقلامی را که آن‌ها پسندیده‌اند به کاربر ورودی توصیه می‌کند. روش‌های متعددی برای یافتن کاربران مشابه وجود دارد (حتی برخی از آن‌ها از یادگیری ماشین استفاده می‌کنند) و روشی که در اینجا از آن استفاده خواهیم کرد بر اساس **تابع همبستگی پیرسون $( \text{Pearson Correlation Function} )$** است

<img src="https://raw.githubusercontent.com/HAKAN-Fatemi/machine-learning-HAKAN/refs/heads/main/file_csv/User_Item.png" width=800px>

فرآیند ایجاد یک سیستم توصیه‌گر مبتنی بر کاربر به شرح زیر است:

*   انتخاب یک کاربر با فیلم‌هایی که کاربر تماشا کرده است
*   بر اساس امتیاز او به فیلم‌ها، پیدا کردن ${ X }$ همسایه‌ی برتر $( \text{top X neighbours} )$
*   دریافت رکورد فیلم‌های تماشا‌شده‌ی کاربر برای هر همسایه
*   محاسبه‌ی نمره‌ی شباهت با استفاده از یک فرمول
*   توصیه‌ی اقلام با بالاترین نمره

بیایید با ایجاد یک کاربر ورودی برای توصیه فیلم به او شروع کنیم

توجه: برای افزودن فیلم‌های بیشتر، کافی است تعداد عناصر موجود در $( \text{userInput} )$ را افزایش دهید. در افزودن فیلم‌های بیشتر آزادید! فقط مطمئن شوید که آن را با حروف بزرگ بنویسید و اگر فیلمی با "$( \text{"The"} )$ شروع می‌شود، مانند $( \text{"The Matrix"} )$، آن را به این شکل بنویسید

 $( \text{"Matrix, The"} )$

In [139]:
userInput = [
            {"title":"Breakfast Club, The", "rating":5},
            {"title":"Toy Story", "rating":3.5},
            {"title":"Jumanji", "rating":2},
            {"title":"Pulp Fiction", "rating":5},
            {"title":"Akira", "rating":4.5}
         ] 
inputMovies = pd.DataFrame(userInput)
inputMovies

,title,rating
0,"Breakfast Club, The",5.0
1,Toy Story,3.5
2,Jumanji,2.0
3,Pulp Fiction,5.0
4,Akira,4.5


#### افزودن $( \text{movieId} )$ به کاربر ورودی

با تکمیل ورودی، بیایید شناسه‌های $( \text{ID} )$ فیلم‌های ورودی را از دیتافریم $( \text{movies} )$ استخراج کرده و آن‌ها را به آن اضافه کنیم

می‌توانیم این کار را با اول فیلتر کردن سطرهایی که شامل عنوان فیلم ورودی هستند و سپس ادغام این زیرمجموعه با دیتافریم ورودی انجام دهیم. همچنین ستون‌های غیرضروری را برای ورودی حذف می‌کنیم تا حافظه آزاد شود

In [140]:
# فیلتر کردن فیلم‌ها بر اساس عنوان
inputId = movies_df[movies_df["title"].isin(inputMovies["title"].tolist())]
# این کار به‌صورت ضمنی بر اساس عنوان ادغام می‌کند movieId. سپس ادغام کردن برای دریافت 
inputMovies = pd.merge(inputId, inputMovies)
# حذف اطلاعاتی که از دیتافریم ورودی استفاده نخواهیم کرد
inputMovies = inputMovies.drop("year", axis=1)
# دیتافریم ورودی نهایی
# اگر فیلمی که در بالا اضافه کردید در اینجا نیست، ممکن است در دیتافریم اصلی نباشد یا املای آن متفاوت باشد، لطفاً بزرگ‌نویسی را بررسی کنید
inputMovies

,movieId,title,rating
0,1,Toy Story,3.5
1,2,Jumanji,2.0
2,296,Pulp Fiction,5.0
3,1274,Akira,4.5
4,1968,"Breakfast Club, The",5.0


#### کاربرانی که فیلم‌های مشابه را دیده‌اند

اکنون با داشتن شناسه‌های $( \text{ID} )$ فیلم‌ها در ورودی خود، می‌توانیم زیرمجموعه‌ای از کاربرانی را دریافت کنیم که فیلم‌های موجود در ورودی ما را تماشا و بررسی کرده‌اند

In [141]:
# فیلتر کردن کاربرانی که فیلم‌های ورودی را تماشا کرده‌اند و ذخیره‌ی آن
userSubset = ratings_df[ratings_df["movieId"].isin(inputMovies["movieId"].tolist())]
userSubset.head()

,userId,movieId,rating
19,4,296,4.0
441,12,1968,3.0
479,13,2,2.0
531,13,1274,5.0
681,14,296,2.0


اکنون سطرها را بر اساس $( \text{user ID} )$ گروه‌بندی می‌کنیم

In [142]:
# گروه‌بندی چندین دیتافریم فرعی ایجاد می‌کند که همگی در ستون مشخص‌شده به‌عنوان پارامتر، مقدار یکسانی دارند.
userSubsetGroup = userSubset.groupby(["userId"])

بیایید به یکی از کاربران نگاه کنیم، برای مثال کاربری با $( \text{userID} = 1130 )$

In [143]:
userSubsetGroup.get_group(1130)

C:\Users\mrb\AppData\Local\Temp\ipykernel_4064\2121309423.py:1: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  userSubsetGroup.get_group(1130)


,userId,movieId,rating
104167,1130,1,0.5
104168,1130,2,4.0
104214,1130,296,4.0
104363,1130,1274,4.5
104443,1130,1968,4.5


بیایید این گروه‌ها را نیز مرتب کنیم تا کاربرانی که بیشترین فیلم‌های مشترک را با ورودی دارند، اولویت بالاتری داشته باشند. این کار توصیه‌ی غنی‌تری ارائه می‌دهد، زیرا نیازی به بررسی تک‌تک کاربران نخواهیم داشت

In [144]:
# مرتب‌سازی به‌طوری که کاربرانی که بیشترین فیلم مشترک را با ورودی دارند، اولویت داشته باشند
userSubsetGroup = sorted(userSubsetGroup, key=lambda x: len(x[1]), reverse=True)

حال بیایید به اولین کاربر نگاه کنیم

In [145]:
userSubsetGroup[0:3]

[((75,),
        userId  movieId  rating
  7507      75        1     5.0
  7508      75        2     3.5
  7540      75      296     5.0
  7633      75     1274     4.5
  7673      75     1968     5.0),
 ((106,),
        userId  movieId  rating
  9083     106        1     2.5
  9084     106        2     3.0
  9115     106      296     3.5
  9198     106     1274     3.0
  9238     106     1968     3.5),
 ((686,),
         userId  movieId  rating
  61336     686        1     4.0
  61337     686        2     3.0
  61377     686      296     4.0
  61478     686     1274     4.0
  61569     686     1968     5.0)]

#### شباهت کاربران به کاربر ورودی

در مرحله‌ی بعد، قصد داریم همه‌ی کاربران (نه واقعاً همه‌!!!) را با کاربر مشخص‌شده‌ی خود مقایسه کرده و کاربری را که بیشترین شباهت را دارد پیدا کنیم
ما قصد داریم میزان شباهت هر کاربر به ورودی را از طریق **ضریب همبستگی پیرسون $( \text{Pearson Correlation Coefficient} )$** محاسبه کنیم. این ضریب برای اندازه‌گیری قدرت یک رابطه‌ی خطی بین دو متغیر استفاده می‌شود. فرمول پیدا کردن این ضریب بین مجموعه‌های $( X )$ و $( Y )$ با $( N )$ مقدار را در تصویر زیر می‌توانید مشاهده کنید

چرا همبستگی پیرسون؟

همبستگی پیرسون نسبت به تغییر مقیاس ناوردا است، یعنی ضرب کردن همه‌ی عناصر در یک ثابت غیرصفر یا افزودن هر ثابتی به همه‌ی عناصر تأثیری در آن ندارد. به‌عنوان مثال، اگر دو بردار $( X )$ و $( Y )$ داشته باشید، آن‌گاه:
$( \text{pearson}(X, Y) = \text{pearson}(X, 2 \times Y + 3) )$.
این ویژگی در سیستم‌های توصیه‌گر بسیار مهم است، زیرا برای مثال دو کاربر ممکن است دو مجموعه‌فیلم را از نظر امتیازهای مطلق کاملاً متفاوت امتیازدهی کنند، اما کاربرانی مشابه (یعنی با ایده‌های مشابه) با امتیازهای مشابه در مقیاس‌های مختلف باشند

![alt text](https://camo.githubusercontent.com/c38f5daec6c87b0f5e73ef68fcfe60dd5c2195e3ab783f717bd741588adb3268/68747470733a2f2f77696b696d656469612e6f72672f6170692f726573745f76312f6d656469612f6d6174682f72656e6465722f7376672f62643163636332393739623066643163316165633936653338366636383661653837346639656330 "Pearson Correlation")

مقادیر داده‌شده توسط فرمول از $( r = -1 )$ تا $( r = 1 )$ متغیر است، که در آن $( 1 )$ یک همبستگی مستقیم بین دو موجودیت را تشکیل می‌دهد (به معنای همبستگی مثبت کامل است) و $( -1 )$ یک همبستگی منفی کامل را تشکیل می‌دهد

در مورد ما، مقدار $( 1 )$ به این معنی است که دو کاربر سلیقه‌های مشابهی دارند، در حالی که مقدار $( -1 )$ به معنای عکس آن است

ما زیرمجموعه‌ای از کاربران را برای پیمایش انتخاب خواهیم کرد. این محدودیت اعمال می‌شود زیرا نمی‌خواهیم با بررسی تک‌تک کاربران، زمان زیادی را تلف کنیم

In [146]:
userSubsetGroup = userSubsetGroup[0:100]

اکنون، همبستگی پیرسون را بین کاربر ورودی و زیرمجموعه‌ی گروه محاسبه کرده و آن را در یک دیکشنری ذخیره می‌کنیم، به‌طوری که کلید $( \text{key} )$، $( \text{userId} )$ و مقدار $( \text{value} )$، ضریب همبستگی باشد

In [147]:
# و مقدار آن ضریب است userId ذخیره‌ی همبستگی پیرسون در یک دیکشنری، که کلید آن 
pearsonCorrelationDict = {}

# برای هر گروه کاربر در زیرمجموعه‌ی ما
for name, group in userSubsetGroup:
    # بیایید با مرتب‌سازی ورودی و گروه کاربر فعلی شروع کنیم تا مقادیر بعداً مخلوط نشوند
    group = group.sort_values(by="movieId")
    inputMovies = inputMovies.sort_values(by="movieId")
    # دریافت N برای فرمول
    nRatings = len(group)
    # دریافت امتیازهای فیلم‌هایی که هر دو مشترک دارند
    temp_df = inputMovies[inputMovies["movieId"].isin(group["movieId"].tolist())]
    # و سپس آن‌ها را در یک متغیر بافر موقت در قالب لیست برای تسهیل محاسبات بعدی ذخیره می‌کنیم
    tempRatingList = temp_df["rating"].tolist()
    # بیایید همچنین نظرات گروه کاربر فعلی را در قالب لیست قرار دهیم
    tempGroupList = group["rating"].tolist()
    # x, y حال بیایید همبستگی پیرسون بین دو کاربر را محاسبه کنیم، به‌اصطلاح
    Sxx = sum([i**2 for i in tempRatingList]) - pow(sum(tempRatingList), 2) / float(nRatings)
    Syy = sum([i**2 for i in tempGroupList]) - pow(sum(tempGroupList), 2) / float(nRatings)
    Sxy = sum(i * j for i, j in zip(tempRatingList, tempGroupList)) - sum(tempRatingList) * sum(tempGroupList) / float(nRatings)
    
    # اگر مخرج مخالف صفر باشد، تقسیم می‌کنیم، در غیر این صورت همبستگی صفر است
    if Sxx != 0 and Syy != 0:
        pearsonCorrelationDict[name] = Sxy / sqrt(Sxx * Syy)
    else:
        pearsonCorrelationDict[name] = 0

In [148]:
pearsonCorrelationDict.items()

dict_items([((75,), 0.8272781516947562), ((106,), 0.5860090386731182), ((686,), 0.8320502943378437), ((815,), 0.5765566601970551), ((1040,), 0.9434563530497265), ((1130,), 0.2891574659831201), ((1502,), 0.8770580193070299), ((1599,), 0.4385290096535153), ((1625,), 0.716114874039432), ((1950,), 0.179028718509858), ((2065,), 0.4385290096535153), ((2128,), 0.5860090386731196), ((2432,), 0.1386750490563073), ((2791,), 0.8770580193070299), ((2839,), 0.8204126541423674), ((2948,), -0.11720180773462392), ((3025,), 0.45124262819713973), ((3040,), 0.89514359254929), ((3186,), 0.6784622064861935), ((3271,), 0.26989594817970664), ((3429,), 0.0), ((3734,), -0.15041420939904673), ((4099,), 0.05860090386731196), ((4208,), 0.29417420270727607), ((4282,), -0.4385290096535115), ((4292,), 0.6564386345361464), ((4415,), -0.11183835382312353), ((4586,), -0.9024852563942795), ((4725,), -0.08006407690254357), ((4818,), 0.4885967564883424), ((5104,), 0.7674257668936507), ((5165,), -0.4385290096535153), ((554

In [149]:
pearsonDF = pd.DataFrame.from_dict(pearsonCorrelationDict, orient="index")
pearsonDF.columns = ["similarityIndex"]
pearsonDF["userId"] = pearsonDF.index
pearsonDF.index = range(len(pearsonDF))
pearsonDF.head()

,similarityIndex,userId
0,0.827278,"(75,)"
1,0.586009,"(106,)"
2,0.832050,"(686,)"
3,0.576557,"(815,)"
4,0.943456,"(1040,)"


#### ۵۰ کاربر برتر مشابه با کاربر ورودی

حال بیایید ۵۰ کاربر برتری را که بیشترین شباهت را به ورودی دارند، دریافت کنیم.

In [150]:
topUsers=pearsonDF.sort_values(by="similarityIndex", ascending=False)[0:50]
topUsers.head()

,similarityIndex,userId
64,0.961678,"(12325,)"
34,0.961538,"(6207,)"
55,0.961538,"(10707,)"
67,0.960769,"(13053,)"
4,0.943456,"(1040,)"


اکنون، بیایید شروع به توصیه‌ی فیلم به کاربر ورودی کنیم.

#### امتیاز کاربران انتخاب‌شده به همه‌ی فیلم‌ها

ما این کار را با گرفتن میانگین وزنی $( \text{weighted average} )$ امتیازهای فیلم‌ها با استفاده از همبستگی پیرسون به‌عنوان وزن انجام خواهیم داد. اما برای انجام این کار، ابتدا باید فیلم‌های تماشا‌شده توسط کاربران موجود در **$( \text{pearsonDF} )** را از دیتافریم $( \text{ratings} )$ دریافت کنیم و سپس همبستگی آن‌ها را در یک ستون جدید به نام $( \text{similarityIndex} )$ ذخیره کنیم. این کار در زیر با ادغام $( \text{merge} )$ این دو جدول انجام می‌شود

In [151]:
# topUsersRating= topUsers.merge(ratings_df, left_on="userId", right_on="userId", how="inner")
# topUsersRating.head()

# فقط بررسی کن که کدام ستون نیاز به تبدیل دارد
if isinstance(topUsers["userId"].iloc[0], tuple):
    topUsers["userId"] = topUsers["userId"].apply(lambda x: x[0])

# ratings_df را تبدیل نکن چون int است
# اما اگر مطمئن نیستی، بررسی کن:
if isinstance(ratings_df["userId"].iloc[0], tuple):
    ratings_df["userId"] = ratings_df["userId"].apply(lambda x: x[0])

# تبدیل به int
topUsers["userId"] = topUsers["userId"].astype(int)
ratings_df["userId"] = ratings_df["userId"].astype(int)

# ادغام
topUsersRating = topUsers.merge(ratings_df, left_on="userId", right_on="userId", how="inner")
topUsersRating.head()

,similarityIndex,userId,movieId,rating
0,0.961678,12325,1,3.5
1,0.961678,12325,2,1.5
2,0.961678,12325,3,3.0
3,0.961678,12325,5,0.5
4,0.961678,12325,6,2.5


اکنون تنها کاری که باید انجام دهیم این است که امتیاز فیلم را در وزن آن (شاخص شباهت) ضرب کنیم، سپس امتیازهای جدید را جمع‌بندی کرده و بر مجموع وزن‌ها تقسیم کنیم

می‌توانیم به‌سادگی این کار را با ضرب دو ستون، سپس گروه‌بندی دیتافریم بر اساس $( \text{movieId} )$ و سپس تقسیم دو ستون انجام دهیم

این ایده‌ی همه‌ی کاربران مشابه را برای فیلم‌های کاندید برای کاربر ورودی نشان می‌دهد

In [152]:
# ضرب شاخص شباهت در امتیاز کاربر
topUsersRating["weightedRating"] = topUsersRating["similarityIndex"] * topUsersRating["rating"]
topUsersRating.head()

,similarityIndex,userId,movieId,rating,weightedRating
0,0.961678,12325,1,3.5,3.365874
1,0.961678,12325,2,1.5,1.442517
2,0.961678,12325,3,3.0,2.885035
3,0.961678,12325,5,0.5,0.480839
4,0.961678,12325,6,2.5,2.404196


In [153]:
# userId پس از گروه‌بندی بر اساس topUsers اعمال جمع بر روی
tempTopUsersRating = topUsersRating.groupby("movieId").sum()[["similarityIndex", "weightedRating"]]
tempTopUsersRating.columns = ["sum_similarityIndex", "sum_weightedRating"]
tempTopUsersRating.head()

,sum_similarityIndex,sum_weightedRating
movieId,,
1,38.376281,140.800834
2,38.376281,96.656745
3,10.253981,27.254477
4,0.929294,2.787882
5,11.723262,27.151751


In [154]:
# ایجاد یک دیتافریم خالی
recommendation_df = pd.DataFrame()
# اکنون میانگین وزنی را محاسبه می‌کنیم
recommendation_df["weighted average recommendation score"] = tempTopUsersRating["sum_weightedRating"] / tempTopUsersRating["sum_similarityIndex"]
recommendation_df["movieId"] = tempTopUsersRating.index
recommendation_df.head()

,weighted average recommendation score,movieId
movieId,,
1,3.668955,1
2,2.518658,2
3,2.657941,3
4,3.000000,4
5,2.316058,5


حال بیایید آن را مرتب کرده و ۲۰ فیلم برتری را که الگوریتم توصیه کرده است، ببینیم

In [155]:
recommendation_df = recommendation_df.sort_values(by="weighted average recommendation score", ascending=False)
recommendation_df.head(10)

,weighted average recommendation score,movieId
movieId,,
3329,5.0,3329
2284,5.0,2284
5073,5.0,5073
28,5.0,28
2848,5.0,2848
136485,5.0,136485
4252,5.0,4252
99,5.0,99
121,5.0,121


In [156]:
movies_df.loc[movies_df["movieId"].isin(recommendation_df.head(10)["movieId"].tolist())]

,movieId,title,year
27,28,Persuasion,1995
97,99,Heidi Fleiss: Hollywood Madam,1995
119,121,"Boys of St. Vincent, The",1992
2200,2284,Bandit Queen,1994
2763,2848,Othello (Tragedy of Othello: The Moor of Venic...,1952
3243,3329,"Year My Voice Broke, The",1987
4159,4252,"Circle, The (Dayereh)",2000
4978,5073,"Son's Room, The (Stanza del figlio, La)",2001
8653,26158,Closely Watched Trains (Ostre sledované vlaky),1966
30005,136485,Robot Chicken: Star Wars,2007


### مزایا و معایب پالایش همکاری‌محور $( \text{Collaborative Filtering} )$

##### مزایا $( \text{Advantages} )$

*   امتیازهای سایر کاربران را در نظر می‌گیرد
*   نیازی به مطالعه یا استخراج اطلاعات از آیتم توصیه‌شده ندارد
*   با علایق کاربر که ممکن است در طول زمان تغییر کند، سازگار می‌شود

##### معایب $( \text{Disadvantages} )$

*   تابع تقریب $( \text{approximation function} )$ می‌تواند کند باشد
*   ممکن است تعداد کاربران برای تقریب کم باشد
*   مسائل مربوط به حریم خصوصی هنگام تلاش برای یادگیری ترجیحات کاربر


<h2>می‌خواهید بیشتر یاد بگیرید؟</h2>

پلتفرم **هوکو** یک بستر جامع تحلیلی و هوش مصنوعی است که مجموعه‌ای از الگوریتم‌های یادگیری ماشین، ابزارهای تحلیل داده و راهکارهای پیش‌بینی هوشمند را در اختیار شما قرار می‌دهد.
این پلتفرم به شما کمک می‌کند تا تصمیم‌های دقیق‌تر، سریع‌تر و مبتنی بر داده بگیرید؛ 

چه به‌صورت فردی، چه در سطح تیمی و یا در مقیاس سازمانی

___

اکنون می‌توانید نسخه آزمایشی رایگان هوکو را فعال کرده و قدرت هوش مصنوعی را در تصمیم‌گیری‌های خود تجربه کنید

 <a href="https://hooko.ir">HOOKO.ir شروع تجربه در </a>

 ___

## Author

Mahdi Fatemi (HAKAN)

Instagram: @Fatemi_303

09220630140

## web
www.hooko.ir

## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2025-11-06 | 1.0  | HAKAN Fatemi  |  ... |

## <h3 align="center"> © HOOKO.IR Corporation. All rights reserved. <h3/>
